# e1 · GARCH and GARCH-MIDAS Evaluation

This notebook evaluates whether the BAA10Y long-run component improves one-month-ahead realised-variance forecasts over the GARCH(1,1) benchmark for XLI, XLB, XLE, XLY and XLF.

It reads the forecast file produced by `b1_garch_garchmidas_forecast_updated.ipynb`. Supplementary diagnostics and the macro-driver selection exercise are reported separately in the appendix.


## Method

**Losses.** MSE-log-RV is $[\log(RV)-\log(F)]^2$. QLIKE is $RV/F-\log(RV/F)-1$. Lower values are preferred for both.

**Tests.** GARCH is nested in GARCH-MIDAS when $\theta=0$. The primary test is therefore one-sided Clark–West (CW) on MSE-log-RV, with the alternative that GARCH-MIDAS improves the forecast. QLIKE uses a one-sided Diebold–Mariano (DM) result in the same direction as supporting evidence only: ordinary DM inference is not formally valid for nested models.

Both use Bartlett Newey–West standard errors with the Andrews bandwidth. For pooled results, the five ETF loss differentials are averaged within month before testing. CW was derived for nested linear OLS forecasts, so its use for maximum-likelihood GARCH-MIDAS is an approximation.


In [1]:
import os
import numpy as np
import pandas as pd
from scipy.stats import norm

FC_DIR = os.path.join(".", "models", "forecasting", "forecasts")
RESULTS_DIR = os.path.join(".", "models", "forecasting", "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

data = pd.read_csv(os.path.join(FC_DIR, "garch_forecasts.csv"), parse_dates=["date", "target_date"])

TICKERS = ["XLI", "XLB", "XLE", "XLY", "XLF"]
G, M = "garch", "garch_midas_baa10y"
MODELS = [G, M]
required = {"model", "etf", "date", "target_date", "actual_logRV", "actual_RV", "fc_log", "fc_level"}
assert required.issubset(data.columns), f"Forecast file is missing {sorted(required - set(data.columns))}"

data = data[data.model.isin(MODELS)].copy()
assert set(data.model) == set(MODELS), "Both model forecasts are required"
assert not data.duplicated(["model", "etf", "date"]).any(), "Duplicate forecast keys"
assert set(data.etf) == set(TICKERS), "Unexpected ETF universe"
assert (data.target_date > data.date).all(), "Every target date must follow its origin"
assert data[["actual_RV", "actual_logRV", "fc_level", "fc_log"]].notna().all().all()
assert (data[["actual_RV", "fc_level"]] > 0).all().all()

n_models = data.groupby(["etf", "date"]).model.nunique()
assert (n_models == len(MODELS)).all(), "The two models do not share a common forecast sample"
actuals = data.pivot(index=["etf", "date"], columns="model", values="actual_RV")
assert (actuals.nunique(axis=1) == 1).all(), "The models disagree on realised targets"

data["MSE_log"] = (data.actual_logRV - data.fc_log) ** 2
ratio = data.actual_RV / data.fc_level
data["QLIKE"] = ratio - np.log(ratio) - 1
data["MSE_level"] = (data.actual_RV - data.fc_level) ** 2


def andrews_bandwidth(x):
    x = np.asarray(x, float); n = len(x); xd = x - x.mean()
    den = float(xd[:-1] @ xd[:-1])
    rho = 0.0 if den <= 0 else float(xd[1:] @ xd[:-1]) / den
    rho = min(max(rho, -0.97), 0.97)
    a1 = 4.0 * rho ** 2 / ((1 - rho) ** 2 * (1 + rho) ** 2)
    return 0 if a1 <= 0 else int(min(np.floor(1.1447 * (a1 * n) ** (1 / 3)), n // 4))


def newey_west_lrv(x, lag):
    x = np.asarray(x, float); n = len(x); xd = x - x.mean()
    lag = max(0, min(lag, n - 1)); s = float(xd @ xd) / n
    for l in range(1, lag + 1):
        s += 2.0 * (1.0 - l / (lag + 1.0)) * float(xd[l:] @ xd[:-l]) / n
    return max(s, 1e-16)


def hac_t(d):
    d = np.asarray(d, float)
    return d.mean() / np.sqrt(newey_west_lrv(d, andrews_bandwidth(d)) / len(d))


def one_sided_p(t):
    return float(1.0 - norm.cdf(t))


## Losses

The models are evaluated on their common ETF-month sample. The full loss table is saved for reporting; the display below shows pooled loss by model.


In [2]:
LOSS_COLUMNS = ["QLIKE", "MSE_log", "MSE_level"]
per_etf = data.groupby(["model", "etf"])[LOSS_COLUMNS].mean().reset_index()
pooled = data.groupby("model")[LOSS_COLUMNS].mean().reset_index().assign(etf="pooled")
losses = pd.concat([per_etf, pooled], ignore_index=True).rename(columns={"etf": "ETF"})
losses.to_csv(os.path.join(RESULTS_DIR, "garch_losses.csv"), index=False)

print(f"Common sample: {data['date'].min():%Y-%m} to {data['date'].max():%Y-%m}; {len(data)} model-ETF-month rows")
display(losses[losses["ETF"] == "pooled"].set_index("model")[LOSS_COLUMNS].round(4))


Common sample: 2008-02 to 2025-11; 2140 model-ETF-month rows


,QLIKE,MSE_log,MSE_level
model,,,
garch,0.2635,0.4736,0.0001
garch_midas_baa10y,0.2682,0.4826,0.0001


## Tests

For each ETF and for the pooled series, CW tests whether GARCH-MIDAS improves MSE-log-RV. DM tests whether GARCH-MIDAS has lower QLIKE, but is supporting evidence only.


In [3]:
PAIR_ID, A_LABEL, B_LABEL = "B1", "GARCH-MIDAS", "GARCH"
loss_idx = losses.set_index(["model", "ETF"])
A = data[data["model"] == M].set_index(["date", "etf"])
B = data[data["model"] == G].set_index(["date", "etf"])
dm_diff = A["QLIKE"] - B["QLIKE"]   # negative favours GARCH-MIDAS
# Clark-West adjusted differential; positive favours GARCH-MIDAS
cw = B["MSE_log"] - (A["MSE_log"] - (B["fc_log"] - A["fc_log"]) ** 2)

rows = []
for etf in ["pooled"] + TICKERS:
    if etf == "pooled":
        dm_t, cw_t = dm_diff.groupby("date").mean(), cw.groupby("date").mean()
    else:
        dm_t, cw_t = dm_diff.xs(etf, level="etf"), cw.xs(etf, level="etf")
    for loss, test, t in [("QLIKE", "DM", -hac_t(dm_t)), ("MSE_log", "Clark-West", hac_t(cw_t))]:
        rows.append({
            "pair": PAIR_ID, "a": A_LABEL, "b": B_LABEL, "ETF": etf, "loss": loss, "test": test,
            "loss_ratio": loss_idx.loc[(M, etf), loss] / loss_idx.loc[(G, etf), loss],
            "gain_t": t, "p_value": one_sided_p(t),
        })

tests = pd.DataFrame(rows)
tests["pct_change"] = (tests["loss_ratio"] - 1) * 100  # negative: GARCH-MIDAS has lower loss
tests.to_csv(os.path.join(RESULTS_DIR, "garch_tests.csv"), index=False)
display(tests[tests.ETF == "pooled"].drop(columns="ETF").round(4).reset_index(drop=True))


,pair,a,b,loss,test,loss_ratio,gain_t,p_value,pct_change
0,B1,GARCH-MIDAS,GARCH,QLIKE,DM,1.0178,-1.7413,0.9592,1.7775
1,B1,GARCH-MIDAS,GARCH,MSE_log,Clark-West,1.0191,1.3270,0.0922,1.9107


## By-sector tests

Each cell reports the GARCH-MIDAS/GARCH loss ratio followed by the one-sided p-value. A ratio below one and a positive test statistic favour GARCH-MIDAS.


In [4]:
SECTORS = {"XLB": "Materials", "XLE": "Energy", "XLF": "Financials", "XLI": "Industrials", "XLY": "Cons. Disc."}


def fmt_p(p):
    return "<0.001" if p < 0.001 else f"{p:.3f}"


def by_sector(loss):
    t = tests[tests["loss"] == loss].copy()
    t["cell"] = [f"{r:.2f} ({fmt_p(p)})" for r, p in zip(t["loss_ratio"], t["p_value"])]
    t["row"] = t["pair"] + ": " + t["a"] + " vs " + t["b"]
    table = t.pivot(index="row", columns="ETF", values="cell")[TICKERS + ["pooled"]]
    table.columns = [f"{c} {SECTORS[c]}" for c in TICKERS] + ["Pooled"]
    return table.rename_axis(None)


for loss, test in [("MSE_log", "Clark-West"), ("QLIKE", "DM")]:
    print(f"{loss}: ratio (p-value), {test}")
    display(by_sector(loss))


MSE_log: ratio (p-value), Clark-West


,XLI Industrials,XLB Materials,XLE Energy,XLY Cons. Disc.,XLF Financials,Pooled
B1: GARCH-MIDAS vs GARCH,1.03 (0.168),1.01 (0.358),1.03 (0.503),1.02 (0.260),1.01 (0.337),1.02 (0.092)


QLIKE: ratio (p-value), DM


,XLI Industrials,XLB Materials,XLE Energy,XLY Cons. Disc.,XLF Financials,Pooled
B1: GARCH-MIDAS vs GARCH,1.03 (0.794),0.99 (0.285),1.03 (0.901),1.02 (0.912),1.02 (0.948),1.02 (0.959)
